# AI Audit Companion: Colab test
Run the cells in order. Before cell 4, add your key in the 🔑 Secrets tab as `GEMINI_API_KEY` and turn on notebook access.

In [ ]:
# 1. Upload ai-audit-companion.zip (the project), then unzip it
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/ai-audit-companion")
print("Working in", os.getcwd()); print(sorted(os.listdir(".")))

In [ ]:
# 2. Install
!pip install -q -r requirements.txt

In [ ]:
# 3. Run the tests (no Gemini quota used)
for t in ["test_zip_to_llm", "test_rules_engine", "test_extraction_parsing", "test_folders"]:
    !python {t}.py 2>&1 | tail -2

In [ ]:
# 4. Give the app your key (kept on the server, never shown on the page)
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Key loaded:", bool(os.environ["GEMINI_API_KEY"]))

In [ ]:
# 5. Start the app and open a public link
import os, re, subprocess, time, urllib.request
subprocess.run(["pkill", "-f", "streamlit run"]); subprocess.run(["pkill", "-f", "cloudflared"])
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=os.environ)
for _ in range(40):
    try:
        urllib.request.urlopen("http://localhost:8501"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("The app did not start:\n" + open("streamlit.log").read()[-2000:])
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(40):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        print("Open this link (if it does not load, wait 30 seconds and reload):\n", m.group(0)); break
    time.sleep(1)
else:
    print("No link yet. Rerun this cell.")

In [ ]:
# If something goes wrong, look at the logs
!tail -20 streamlit.log; echo ---; tail -5 audit_app.log 2>/dev/null